In [ ]:
from pathlib import Path
from PIL import Image
from pixel_arena.dataset_utils.celeb_a_mask_hq import get_prompt
from google.genai import types
from io import BytesIO
import pickle

In [ ]:
def to_pil_image(image: types.Image) -> Image.Image:
    bytes = image.image_bytes
    return Image.open(BytesIO(bytes))

## Test

In [ ]:
from clients import gemini_clients

original_image_path = Path(
    "eval-set/celeb/images-150/0700985d198843cbb080fbb525d44012.jpg"
)
color_palette_path = Path("label_palettes/seg_labels_celeb.png")
label_colors = None

MODEL = "gemini-3-pro-image-preview"
client = gemini_clients[0]

In [ ]:
original_file_name = original_image_path.stem


contents = [
    # first image is the original image
    Image.open(original_image_path).convert("RGB"),
    # second image is the color palette, as mentioned in the prompt
    Image.open(color_palette_path).convert("RGB"),
    get_prompt(label_colors),
]

thinking_config = (
    types.ThinkingConfig(include_thoughts=True)
    if MODEL == "gemini-3-pro-image-preview"
    else None
)
image_size = "1K" if MODEL == "gemini-3-pro-image-preview" else None

thinking_process = []
results = []


response = await client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=types.GenerateContentConfig(
        temperature=1.0,
        response_modalities=[
            "IMAGE",
            "TEXT",
        ],
        image_config=types.ImageConfig(
            aspect_ratio="1:1",
            image_size=image_size,
        ),
        top_p=0.95,
        thinking_config=thinking_config,
    ),
)

for part in response.parts:
    if part.thought:
        if part.text:
            thinking_process.append(part.text)
        elif image := part.as_image():
            thinking_process.append(image)
    if part.text is not None:
        results.append(part.text)
    elif image := part.as_image():
        results.append(image)
    else:
        pass

display(to_pil_image(thinking_process[1]))
display(to_pil_image(results[1]))

In [ ]:
print(thinking_process)

In [ ]:
print(results)

In [ ]:
with open(
    "saved_result_binary/error_cases/response_of_error_case_partial_wrong_left_right.pkl",
    "wb",
) as f:
    pickle.dump(response, f)

## Load

In [ ]:
path = "saved_result_binary/error_cases/response_of_error_case_partial_wrong_left_right.pkl"

with open(path, "rb") as f:
    response = pickle.load(f)


thinking_process = []
results = []

for part in response.parts:
    if part.thought:
        if part.text:
            thinking_process.append(part.text)
        elif image := part.as_image():
            thinking_process.append(image)
    if part.text is not None:
        results.append(part.text)
    elif image := part.as_image():
        results.append(image)
    else:
        pass

print(thinking_process[2])
display(to_pil_image(thinking_process[1]))
display(to_pil_image(results[1]))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import math

cases = {
    "partially wrong left and right": "saved_result_binary/error_cases/response_of_error_case_partial_wrong_left_right.pkl",
    "wrong annotation for hand": "saved_result_binary/error_cases/response_of_error_case_wrong_annotation_for_hand.pkl",
}

reference_mask_path = "eval-set/celeb/masks-1024/0700985d198843cbb080fbb525d44012.png"


def get_result_image(pkl_path):
    with open(pkl_path, "rb") as f:
        resp = pickle.load(f)
    # Collect all images that are NOT part of the thought process
    imgs = []
    for part in resp.parts:
        if not part.thought and (img := part.as_image()):
            imgs.append(img)
    if imgs:
        return to_pil_image(imgs[0])
    return None


def draw_palette(ax, labels, colors, columns=2):
    total_items = len(labels)
    rows = math.ceil(total_items / columns)

    card_size = 1.0
    label_offset = 0.2
    cell_width = card_size + 2.5  # Give more space for text
    cell_height = card_size + 0.5
    side_margin = 0.5
    top_margin = 0.5

    palette_width = side_margin * 2 + columns * cell_width
    palette_height = top_margin + rows * cell_height

    ax.set_xlim(0, palette_width)
    ax.set_ylim(0, palette_height)
    ax.invert_yaxis()
    ax.axis("off")

    for idx, (label, color) in enumerate(zip(labels, colors)):
        row = idx // columns
        col = idx % columns

        x = side_margin + col * cell_width
        y = top_margin + row * cell_height

        normalized_color = [c / 255.0 for c in color]
        rect = Rectangle(
            (x, y),
            card_size,
            card_size,
            facecolor=normalized_color,
            edgecolor="black",
            linewidth=1,
        )
        ax.add_patch(rect)

        label_x = x + card_size + 0.2
        label_y = y + card_size / 2
        ax.text(
            label_x,
            label_y,
            label,
            ha="left",
            va="center",
            fontsize=10,
            color="black",
            weight="bold",
        )


# Load reference images
try:
    orig_img = Image.open(original_image_path).convert("RGB")
except NameError:
    orig_img = Image.open(
        "eval-set/celeb/images-150/0700985d198843cbb080fbb525d44012.jpg"
    ).convert("RGB")

ref_mask = Image.open(reference_mask_path).convert("RGB")

# Plotting
fig, axs = plt.subplots(2, 3, figsize=(15, 10))
axs = axs.flatten()

# 1. Original Image
axs[0].imshow(orig_img)
axs[0].set_title("Original Image")
axs[0].axis("off")

# 2. Reference Mask
axs[1].imshow(ref_mask)
axs[1].set_title("Reference Mask")
axs[1].axis("off")

# 3. Palette
target_labels = ["left_eye", "right_eye", "left_eyebrow", "right_eyebrow", "cloth"]
target_colors = [
    [51, 51, 255],
    [204, 0, 204],
    [0, 255, 255],
    [255, 204, 204],
    [0, 204, 0],
]
draw_palette(axs[2], target_labels, target_colors, columns=1)
axs[2].set_title("Key Labels")

# 4-6. Error cases
for i, (title, pkl_path) in enumerate(cases.items()):
    img = get_result_image(pkl_path)
    if img:
        axs[i + 3].imshow(img)
        axs[i + 3].set_title(title)
    axs[i + 3].axis("off")

axs[5].axis("off")
plt.tight_layout()
plt.show()